FEATURE 1 - THE CHAT PARSER

In [1]:
#opening file
with open('hostel_bois.txt','r',encoding='utf-8')as f:
  content=f.read()


splitting up the messages

In [2]:
system_messages=0
media_omitted=0
deleted_messages=0
messages=[]


lines=content.split("\n")

for line in lines:
  parts=line.split(" - ",1)#timeline and sending part are seperated
  if ":" in parts[1]:
    sender=parts[1].split(":",1)#sender name and message are seperated
    message={
    "timestamp":parts[0].strip(),
    "sender":sender[0].strip(),
    "messaage":sender[1].strip(),
    }#dictionary created
    messages.append(message)#appended each dictinory into messages
    if "<Media omitted>" in sender[1]:
      media_omitted += 1
    if "This message was deleted" in sender[1]:
      deleted_messages += 1
  else:
    system_messages+=1
participants=list(set([message["sender"] for message in messages]))#list of participants

days=list(set([message["timestamp"].split(",")[0] for message in messages]))#list of days

print(f"Succesfully parsed {len(lines)} messages from {len(participants)} participants over {len(days)} days, skipped {system_messages} system messages, {media_omitted} media omitted, {deleted_messages} deleted messages")




Succesfully parsed 3178 messages from 6 participants over 60 days, skipped 4 system messages, 32 media omitted, 15 deleted messages


FEATURE 2 - GROUP OVERVIEW

In [3]:
#group name
lines=content.split("\n")
for line in lines:
  if "created group" in line:
    group_name=line.split('created group',1)
    group_name=group_name[1].strip(' " ')


#period
months=['January','February','March','April','May','June','July','August','September','October','November','December']
start_time=messages[0]['timestamp'].split(',')[0].split('/')
start_time[1]=months[int(start_time[1])-1]
end_time=messages[-1]['timestamp'].split(',')[0].split('/')
end_time[1]=months[int(end_time[1])-1]

period=(f"{start_time[0]} {start_time[1]} {start_time[2]} to {end_time[0]} {end_time[1]} {end_time[2]} ({len(days)} days)")


#counting each person messages
message_counter={}
for message in messages:
  sender=message["sender"]
  if sender in message_counter:
    message_counter[sender]+=1
  else:
    message_counter[sender]=1
message_counter=list(message_counter.items())


#arrange in decreasing order of number of messages per person
for i in range (len(message_counter)):
  for j in range (len(message_counter)-1-i):
    if message_counter[j][1]<message_counter[j+1][1]:
      message_counter[j+1],message_counter[j]=message_counter[j],message_counter[j+1]




#calculating message per person in percentage
counter=[]
for i in range (len(message_counter)):
  sender=message_counter[i][0]
  count=message_counter[i][1]
  percentage=(message_counter[i][1]/len(messages))*100
  counter.append(f"{sender:<16} :{count:>2} ({round(percentage,1)}%)")



#printing
print("="*60)
print(f"{'GROUP OVERVIEW':^60}")
print("="*60)
print(f"{'Group':<16}: {group_name}")
print(f"{'period':<16}:{period}")
print(f"{'Total messages':<16}:{len(messages)}")
print(f"{'Participants':<16}:{len(participants)}")
print('\nMESSAGES PER PERSON')
for line in counter:
  print(line)

                       GROUP OVERVIEW                       
Group           : Hostel Bois 4ever
period          :01 April 24 to 30 May 24 (60 days)
Total messages  :3174
Participants    :6

MESSAGES PER PERSON
Rahul            :953 (30.0%)
Priya            :718 (22.6%)
Neha             :635 (20.0%)
Aman             :490 (15.4%)
Karan            :354 (11.2%)
Vikas            :24 (0.8%)


FEATURE 3 - MOST ACTIVE DAY & HOUR

In [4]:
#finding busiest day
day_counter={}
for message in messages:
  day=message['timestamp'].split(",")[0]
  if day not in day_counter:
    day_counter[day]=1
  else:
    day_counter[day]+=1
busiest_day=max(day_counter,key=day_counter.get)
busy_msg=day_counter[busiest_day]#number of messages in busiest day
#to change month in name
busiest_day=busiest_day.split("/")
busiest_day[1]=months[int(busiest_day[1])-1]

print(f"{'Busiest day':<16}: {busiest_day[0]} {busiest_day[1]} {busiest_day[2]:<4}({busy_msg} messages)")


#finding busiest hour
hour_counter={}
for message in messages:
  hour=message['timestamp'].split(",")[1].split(":")[0]
  if hour not in hour_counter:
    hour_counter[hour]=1
  else:
    hour_counter[hour]+=1

busiest_hour=max(hour_counter,key=hour_counter.get)
hour_msg=hour_counter[busiest_hour]#number of messages at busiest hour
avg_msg=int(hour_msg/len(days))#avg messages at busiest hour per day

print(f"{'Busiest hour':<16}:{busiest_hour}:00 - {int(busiest_hour)+1}:00 (avg {avg_msg} messages per day)")





Busiest day     : 04 May 24  (76 messages)
Busiest hour    : 18:00 - 19:00 (avg 4 messages per day)


FEATURE 4- ACTIVITY HEATMAP(NUMPY)

In [5]:
import numpy as np
matrix=np.zeros((len(participants),24),dtype=int)#create a matrix to store data

#names cannot be directly called by a matrix. so it is coverted into a callable form
#eg: 'rahul': 0
person_index={}
for i in range (len(participants)):
  person_index[participants[i]]=i


for message in messages:
  sender=message['sender']
  row=person_index[sender]
  hour=int(message['timestamp'].split(",")[1].split(":")[0])
  matrix[row][hour]+=1

print("\nACTIVE HEATMAP (hours of day,columns 00 to 23)")
print("        ",end=" ")
for i in range(24):
  print(f"{i:<3}",end=(" "))
print("")# printing hours

for i in range(len(participants)):
  print(f"\n{participants[i]:<8}",end=" ")# printing partcipants name
  for j in range(24):
    #print(f"{matrix[i][j]}",end=" ")
    max_msg=np.max(matrix[i])#max no of msgs of each person
    value = (matrix[i][j]/max_msg)*100 #percentage value relative to person's maximum
    if value <= 25:
        print(f"{'.':<2}", end="  ")
    elif value <= 50:
        print(f"{'▒':<2}", end="  ")
    elif value <= 75:
        print(f"{'▓':<2}", end="  ")
    else:
        print(f"{'█':<2}", end="  ")


print()







ACTIVE HEATMAP (hours of day,columns 00 to 23)
         0   1   2   3   4   5   6   7   8   9   10  11  12  13  14  15  16  17  18  19  20  21  22  23  

Rahul    .   .   .   .   .   .   .   .   .   .   .   .   ▓   ▒   ▒   ▓   ▓   ▒   █   ▓   ▒   █   ▓   ▓   
Priya    .   .   .   .   .   .   .   ▒   ▓   █   █   █   █   ▓   ▓   ▒   ▒   ▓   ▓   █   ▓   ▒   ▒   .   
Neha     .   .   .   .   .   ▒   .   .   ▓   █   █   ▒   ▓   ▓   ▒   .   ▓   █   █   █   ▓   ▒   ▒   ▒   
Aman     ▓   █   ▓   ▓   █   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   ▓   
Karan    .   .   .   .   .   .   .   .   ▒   ▒   ▓   ▒   █   ▓   █   ▓   ▓   ▓   ▓   █   ▓   ▒   .   .   
Vikas    .   .   .   .   .   .   .   ▒   █   ▒   ▒   .   ▓   ▓   .   ▒   ▒   █   ▓   ▓   ▒   ▒   ▒   ▓   


FEATURE 5 - TOP WORDS

In [6]:
word_counter={}#store each word
for message in messages:
  message_lower=message["messaage"].lower()#convert to lower case
  if message_lower=="<media omitted>":
    continue
  if message_lower=="this message was deleted":
    continue
  words=message_lower.split(" ")#split each message into words
  for word in words:
    word=word.strip(".,!?():;\"'")
    stop_words=[ "the","is","a","an","to","of","and","or","for","in","on","i", "are", "was", "were","am", "i", "me", "my", "you", "your","he", "she", "it", "we", "they","this", "that", "these", "those","to", "of", "in", "on", "at", "for","with", "from", "by", "as","and", "or", "but","how", "what", "when", "where", "why","so", "about","today","hai","today","his","have",'just',"which","everyone",'telling','up','one','had','started','no','entire','please','anyone']
    if word not in stop_words:
      if word not in word_counter:
        word_counter[word]=1
      else:
        word_counter[word]+=1
word_counter=list(word_counter.items())

#arrange no of messages in descending order
for i in range (len(word_counter)):
  for j in range (len(word_counter)-1-i):
    if word_counter[j][1]<word_counter[j+1][1]:
      word_counter[j+1],word_counter[j]=word_counter[j],word_counter[j+1]

print("\nTHIS GROUP'S FAVOURITE WORDS")

#render each topword with horizontal bar
for i in range (0,5):
  top_words=word_counter[i][0]
  counts=word_counter[i][1]
  bar="█"*(counts//10)
  print(f"{top_words:<16} | {bar}({counts:>2})\n")






THIS GROUP'S FAVOURITE WORDS
guys             | ███████████████████████████████(318)

bhai             | ████████████████(160)

scene            | ██████████████(145)

yaar             | █████████████(139)

kya              | █████████████(133)



FEATURE 6 - RESPONSE SPEED & SILENT STREAKS

In [7]:
from datetime import datetime


# AVERAGE RESPOND TIME

response_time={}# total respond time
response_counter={}#total no of responses


for i in range(len(messages)):
    current_sender=messages[i]['sender']
    t1=datetime.strptime(messages[i]['timestamp'],"%d/%m/%y, %H:%M")#replier time
    j=i-1
    while j>=0 and messages[j]['sender']!=current_sender:
      t2=datetime.strptime(messages[j]['timestamp'],"%d/%m/%y, %H:%M")#sender time
      diff=(t1-t2).total_seconds() #timetaken to respond(in seconds)
      if diff > 3600: #ignore gap greater than 1 hour
        j-=1
        continue

      if current_sender not in response_time:
        response_time[current_sender]=diff
      else:
        response_time[current_sender]+=diff

      j-=1

    if current_sender not in response_counter:
      response_counter[current_sender] = 1
    else:
      response_counter[current_sender] += 1


avg_time={}
for person in response_time:
    avg_time[person]=(response_time[person]/response_counter[person])/60

fastest = min(avg_time, key=avg_time.get)
slowest = max(avg_time, key=avg_time.get)

print("\nRESPONSE PATTERNS")
print(f"Fastest replier : {fastest} ({avg_time[fastest]:.2f} minutes)")
print(f"Slowest replier : {slowest} ({avg_time[slowest]:.2f} minutes)")

#LONGEST SILENT STREAKS

from datetime import datetime

last_message = {}
longest_streak = {}

for message in messages:
  sender=message['sender']
  current_time=datetime.strptime(message['timestamp'],"%d/%m/%y, %H:%M")
  if sender not in last_message:
    last_message[sender]=current_time
  else:
    silent=(current_time-last_message[sender]).total_seconds()#silent time
    last_message[sender]=current_time
    if sender not in longest_streak:
      longest_streak[sender] = silent
    elif silent > longest_streak[sender]:
      longest_streak[sender] = silent


print("\nLONGEST SILENT STREAKS")


sorted_streaks= sorted(longest_streak.items(), key=lambda x: x[1], reverse=True)
for person, streak in sorted_streaks:
    days = streak / (60 * 60 * 24)
    print(f"{person:<10} : {int(days)} days")


RESPONSE PATTERNS
Fastest replier : Rahul (11.01 minutes)
Slowest replier : Vikas (82.42 minutes)

LONGEST SILENT STREAKS
Vikas      : 12 days
Neha       : 1 days
Rahul      : 1 days
Aman       : 0 days
Karan      : 0 days
Priya      : 0 days


FEATURE 7-PERSONALITY ARCHETYPE DETECTION

In [8]:
scores={}
for person in participants:
  scores[person]={}

#spammer

burst_data = {}

for person in participants:
    burst_data[person] = []

curent_sender = None
burst_count = 0

for message in messages:
    sender = message["sender"]
    if sender == curent_sender:
        burst_count += 1
    else:
        if curent_sender is not None:
            burst_data[curent_sender].append(burst_count)
        curent_sender = sender
        burst_count = 1
if curent_sender is not None:
    burst_data[curent_sender].append(burst_count)
avg_burst={}
for person in participants:
    avg_burst[person]=sum(burst_data[person])/len(burst_data[person])
    scores[person]["THE SPAMMER"] = round(avg_burst[person],1)



#GROUP MOM
caring_words = [
    "okay",
    "safe",
    "eat",
    "sleep",
    "take care",
    "are you",
    "please",
    "reminder",
    "drink water",
    "don't forget"
]

caring_count = {}
total_count= {}
for person in participants:
    caring_count[person] = 0
    total_count[person] = 0

for person in participants:
    caring_count[person] = 0
for message in messages:
    sender = message["sender"]
    total_count[sender]+=1
    text = message["messaage"].lower()
    for word in caring_words:
      word=word.lower()
      if word in text:
        caring_count[sender]+=1
        break
for person in participants:
  scores[person]["THE GROUP MOM"]=(caring_count[person]/total_count[person])*100
  scores[person]["THE GROUP MOM"]=round(scores[person]["THE GROUP MOM"],1)



#NIGHT OWL
night_count = {}
total_ncount = {}

for person in participants:
    night_count[person] = 0
    total_ncount[person] = 0
for message in messages:
  sender=message["sender"]
  hour=int(message["timestamp"].split(",")[1].split(":")[0])
  total_ncount[sender]+=1
  if hour == 23 or hour < 5:
    night_count[sender]+=1
for person in participants:
  scores[person]["THE NIGHT OWL"]=(night_count[person]/total_ncount[person])*100
  scores[person]["THE NIGHT OWL"]=round(scores[person]["THE NIGHT OWL"],1)



#STORY TELLER



word_count = {}# total word send
message_count = {}#total message send

for person in participants:
    word_count[person] = 0
    message_count[person] = 0

for message in messages:
  sender=message["sender"]
  message_count[sender]+=1
  word=message['messaage'].split()
  word_count[sender]+=len(word)
story_teller={}
for person in participants:
  story_teller[person]=(word_count[person]/message_count[person])
  scores[person]["THE STORY TELLER"]=round(story_teller[person],1)


#DRAMA QUEEN

drama_count = {}#total dramatic msgs send
total_dcount = {}#total msgs send

for person in participants:
    drama_count[person] = 0
    total_dcount[person] = 0

for message in messages:
  sender=message["sender"]
  total_dcount[sender]+=1
  text=message["messaage"]
  if text.count("!")>=2:
    drama_count[sender]+=1
  else:
        letters = ""

        for ch in text:
            if ch.isalpha():
                letters += ch

        if len(letters) >= 3 and letters.isupper():
            drama_count[sender] += 1
drama_queen={}
for person in participants:
  drama_queen[person]=(drama_count[person]/total_dcount[person])*100
  scores[person]["THE DRAMA QUEEN"]=round(drama_queen[person],1)


#GHOST

days=list(set([message["timestamp"].split(",")[0] for message in messages]))

active_days = {}
silent_days = {}

for person in participants:
    active_days[person] = set()
for message in messages:
  sender=message["sender"]
  day=message["timestamp"].split(",")[0]
  if day not in active_days[sender]:
    active_days[sender].add(day)

ghost={}
for person in participants:
  silent_days[person]=len(days)-len(active_days[person])
  ghost[person]=(silent_days[person]/len(days))*100
  scores[person]["THE GHOST"]=round(int(ghost[person]))

#QUESTION MASTER

question_count = {}
total_qcount = {}

for person in participants:
    question_count[person] = 0
    total_qcount[person] = 0
for message in messages:

    sender = message["sender"]
    text = message["messaage"].strip()

    total_qcount[sender] += 1
    if text.endswith("?"):
        question_count[sender] += 1
question_master={}
for person in participants:
  question_master[person]=(question_count[person]/total_qcount[person])*100
  scores[person]["THE QUESTION MASTER"]=round(question_master[person],1)

#print(scores)



print("\nPERSONALITY ARCHETYPES\n")

for person in participants:

    best_archetype = max(scores[person], key=scores[person].get)
    best_score = scores[person][best_archetype]
    if best_archetype == "THE SPAMMER":
        print(f"{person:<8} -> {best_archetype:<18} (avg {best_score:.1f} msgs in a row)")

    elif best_archetype == "THE GROUP MOM":
        print(f"{person:<8} -> {best_archetype:<18} ({best_score:.1f}% caring keywords)")

    elif best_archetype == "THE NIGHT OWL":
        print(f"{person:<8} -> {best_archetype:<18} ({best_score:.1f}% msgs after 11 PM)")

    elif best_archetype == "THE STORY TELLER":
        print(f"{person:<8} -> {best_archetype:<18} (avg {best_score:.1f} words/msg)")

    elif best_archetype == "THE DRAMA QUEEN":
        print(f"{person:<8} -> {best_archetype:<18} ({best_score:.1f}% dramatic msgs)")

    elif best_archetype == "THE GHOST":
        print(f"{person:<8} -> {best_archetype:<18} (silent {best_score:.1f}% of days)")

    elif best_archetype == "THE QUESTION MASTER":
        print(f"{person:<8} -> {best_archetype:<18} ({best_score:.1f}% questions)")






PERSONALITY ARCHETYPES

Rahul    -> THE NIGHT OWL      (13.4% msgs after 11 PM)
Priya    -> THE GROUP MOM      (63.9% caring keywords)
Neha     -> THE DRAMA QUEEN    (62.2% dramatic msgs)
Aman     -> THE NIGHT OWL      (79.8% msgs after 11 PM)
Karan    -> THE STORY TELLER   (avg 55.7 words/msg)
Vikas    -> THE GHOST          (silent 73.0% of days)
